In [ ]:
# This notebook sets up and trains Deepdisc (Merz et al. 2023) on the Cen V854 plates, as a test run.

# [Setup]: 

# Ran the following in Anaconda Prompt:
# git clone https://github.com/burke86/deepdisc.git
# cd deepdisc
# conda create -n deepdisc python=3.10 -y
# conda activate deepdisc
# pip install pybind11
# NOTE: scarlet skipped, does not compile on Windows (optional dependency, not needed for detection)

# Ran the following in Anaconda Prompt:
# nvidia-smi

# If Version CUDA is not 12.1, install older versions to be compatible with torch and relevant packages:
# Ran the following in Anaconda Prompt:
# conda activate deepdisc
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Ran the following in Anaconda Prompt:
# pip install --no-build-isolation git+https://github.com/facebookresearch/detectron2.git
# mkdir taufit
# echo __version__ = "0.1.0" > taufit\version.py
# echo. > requirements.txt
# pip install --no-build-isolation -e .
# pip install --no-deps --no-build-isolation .
# pip install opencv-python
# pip install astropy photutils matplotlib pandas ipykernel
# python -m ipykernel install --user --name deepdisc --display-name "Python (deepdisc)"

# Now, in VSCodium, click top right for environment -> "Choose another Kernel" -> "deepdisc"

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\deepdisc\src")

from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import find_peaks
import numpy as np
import cv2
import json
import os

cutout_dir = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\raw\v854_cen_dasch\cutouts")
output_dir = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\deepdisc")
img_dir = output_dir / "images"
os.makedirs(img_dir, exist_ok=True) # Create folder if does not exist already

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts")

THRESHOLD_SIGMA = 5.0
BOX_SIZE = 11
BOX_HALF = 8  # half-size of bounding box around each source

coco = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "star"}]
}

ann_id = 0

for img_id, fits_path in enumerate(cutouts):
    # Load and normalize
    data = fits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    data_sub = data - median

    # Detect sources
    sources = find_peaks(data_sub, threshold=THRESHOLD_SIGMA * std, box_size=BOX_SIZE)
    if sources is None or len(sources) == 0:
        continue

    # Normalize image to 8-bit PNG
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)

    png_name = fits_path.stem + ".png"
    png_path = img_dir / png_name
    cv2.imwrite(str(png_path), img_8bit)

    h, w = img_8bit.shape
    coco["images"].append({
        "id": img_id,
        "file_name": str(png_path),
        "height": h,
        "width": w
    })

    # Add annotations
    for row in sources:
        x, y = float(row["x_peak"]), float(row["y_peak"])
        x1 = max(0, x - BOX_HALF)
        y1 = max(0, y - BOX_HALF)
        bw = min(BOX_HALF * 2, w - x1)
        bh = min(BOX_HALF * 2, h - y1)

        coco["annotations"].append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": 1,
            "bbox": [x1, y1, bw, bh],
            "area": bw * bh,
            "iscrowd": 0
        })
        ann_id += 1

# Save COCO JSON
json_path = output_dir / "annotations.json"
with open(json_path, "w") as f:
    json.dump(coco, f)

print(f"Saved {len(coco['images'])} images and {len(coco['annotations'])} annotations")
print(f"JSON: {json_path}")

Found 0 cutouts
Saved 0 images and 0 annotations
JSON: C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data\deepdisc\annotations.json
